# SECOND ATTEMPT

### Cyber Attack Detection & Risk Prediction Dataset
Overview
The Cyber Attack Detection & Risk Prediction Dataset is a realistic synthetic dataset containing 100,000+ enterprise cybersecurity incidents. It simulates complete cyber attack lifecycles, from initial access and attacker behavior to incident response, financial impact, and overall risk assessment.
Designed for machine learning, data analysis, and cybersecurity research, this dataset includes realistic relationships between security controls, vulnerabilities, attack progression, and business impact, making it suitable for both academic and industry projects.

In [37]:
# Imports
%matplotlib inline
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
import imblearn
import keras
import random
import tensorflow as tf

seed = 7
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)

# Libraries for splitting, scaling, encoding and feature selection
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier


# Libraries for models
from sklearn.svm import SVC
from sklearn.naive_bayes import BernoulliNB
from sklearn import tree
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical
from xgboost import XGBClassifier


from imblearn.over_sampling import SMOTE

from sklearn import metrics
from sklearn.model_selection import cross_val_score

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Settings
# pd.set_option('display.max_columns', None)
# import sys
# #np.set_printoptions(threshold=np.nan)
# np.set_printoptions(threshold=sys.maxsize)
# np.set_printoptions(precision=3)
# sns.set(style="darkgrid")
# plt.rcParams['axes.labelsize'] = 14
# plt.rcParams['xtick.labelsize'] = 12
# plt.rcParams['ytick.labelsize'] = 12

In [ ]:
#command used to mount drive
from google.colab import drive
drive.mount('/content/drive')



### 1. Load and Preview The Data


In [ ]:
# I will be starting with the Enterprise_Cyber_Kill_Chain_Dataset data which is not partitioned yet
ECKCD_dataset = pd.read_csv('/content/drive/MyDrive/solutions/Enterprise_Cyber_Kill_Chain_Dataset.csv')
print(ECKCD_dataset.head())

### 2. EDA

In [ ]:
# I will then preview the dataset, show its shape and summary statistics
print("The dataset has {} rows and {} columns".format(ECKCD_dataset.shape[0],ECKCD_dataset.shape[1]))
ECKCD_dataset.describe()

In [ ]:
print(ECKCD_dataset.columns.tolist())
# From the result, we see that this dataset has so many features that it can be used for both prosspective or retrospective prediction
# I will be going the prospective route, i.e. predicting attacks before an incident occurs
# It has many columns that may not be necessary for the prediction of attacks itself
# Rather, they are dependent on the occurence of attacks such as: Incident_Severity, Incident_ID, Attack_Vector
# It is important to note that in this dataset, each row represents an inciddent (successsful or unsuccessful attempt)
# I will also choosse my Y variable here to be 'Risk_Level' rather than 'Cyber_Risk_Score' for prediction ease and accuracy
# I will then drop the columns I do not need to avoid data leakage

# # identifiers / timing — not posture
#     'Incident_ID', 'Timestamp', 'Hour', 'DayOfWeek', 'Month',
#     'Business_Hours', 'Weekend',

#     # attack characteristics — not knowable before a specific attack happens
#     'Attack_Vector', 'Threat_Actor', 'Attack_Stage', 'Attack_Complexity',
#     'Attack_Success', 'Zero_Day',

#     # attack progression/behavior — only exist once an attack is underway
#     'Phishing_Click', 'Credential_Stolen', 'Privilege_Escalation',
#     'Lateral_Movement', 'Persistence', 'Data_Encrypted',

#     # impact/outcome metrics — measured after the incident
#     'Detection_Time_Min', 'Response_Time_Min', 'Downtime_Hours',
#     'Records_Compromised', 'Financial_Loss_USD', 'Recovery_Cost_USD',
#     'Data_Exfiltration_GB',

#     # leakage — Risk_Level is derived from these
#     'Cyber_Risk_Score', 'Incident_Severity'

##### Initial Feature Dropping

In [ ]:
# specify features to be droppedd and drop them

features_to_drop = ['Incident_ID', 'Timestamp', 'Hour', 'DayOfWeek', 'Month',
    'Business_Hours', 'Weekend', 'Attack_Vector', 'Threat_Actor', 'Attack_Stage', 'Attack_Complexity',
    'Attack_Success', 'Zero_Day', 'Phishing_Click', 'Credential_Stolen', 'Privilege_Escalation',
    'Lateral_Movement', 'Persistence', 'Data_Encrypted',     'Detection_Time_Min', 'Response_Time_Min', 'Downtime_Hours',
    'Records_Compromised', 'Financial_Loss_USD', 'Recovery_Cost_USD',
    'Data_Exfiltration_GB', 'Cyber_Risk_Score', 'Incident_Severity']
data = ECKCD_dataset.drop(columns=features_to_drop)



In [ ]:
# Investigate Class Imbalance
print(data['Risk_Level'].value_counts())
# See it in percentage
print(data['Risk_Level'].value_counts(normalize=True)*100)

### 3. Data Cleaning - Missing, Infinite and Duplicate Values

In [ ]:
# Null Values
null_values = data.isnull().sum()
print(null_values[null_values>0])
# we see there are 42,983 missing values in this dataset, all belonging to the Compliance column
data['Compliance'] = data['Compliance'].fillna('None')

# Infinite Values
numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns
inf_values = np.isinf(data[numeric_cols]).sum()
print(inf_values)
# No infinite values

# Duplicate Values
duplicate_values = data.duplicated().sum()
print(duplicate_values) # They were 500, I will drop them below
data = data.drop_duplicates()

### 4. Encoding Categorical Columns in Independent Variables (X)

In [ ]:
# Define independent and dependent variables
x = data.drop(columns=['Risk_Level'])
y = data['Risk_Level']


# define non-numeric columns
cat_cols = x.select_dtypes(include=['object']).columns
print(cat_cols)

# Encode categorical variables
x = pd.get_dummies(x, columns=cat_cols, drop_first=True)
print(x.shape)

### 5. Train/Test Split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    x, y, train_size=0.8, random_state=7, stratify=y
)

# Addded this afterwardd to handle imbalalance
smote = SMOTE(random_state=7)
x_train, y_train = smote.fit_resample(x_train, y_train)

### 6. Scale Numerical Data

In [ ]:
scaler = StandardScaler()
num_cols = x.select_dtypes(include=['int64', 'float64']).columns
dummy_cols = [c for c in x.columns if c not in num_cols]

sc_train = scaler.fit_transform(x_train[num_cols])
sc_test = scaler.transform(x_test[num_cols])

sc_traindf = pd.DataFrame(sc_train, columns=num_cols, index=x_train.index)
sc_testdf = pd.DataFrame(sc_test, columns=num_cols, index=x_test.index)

### 7. Encode Dependent Variable (Y)

In [ ]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [ ]:
# I am reassigning dataframes to variables with simpler names for ease

X = pd.concat([sc_traindf, x_train[dummy_cols].astype(int)], axis=1)
Y = y_train_encoded.copy()
X_TEST = pd.concat([sc_testdf, x_test[dummy_cols].astype(int)], axis=1)
Y_TEST = y_test_encoded.copy()

### 8. Feature Selection

In [ ]:
# Define and train the feature classifier
rfc = RandomForestClassifier(random_state=7)
rfc.fit(X, Y)

# Create a dataframe showing features against their importance score
importances = pd.DataFrame({
    'feature': X.columns,
    'importance': np.round(rfc.feature_importances_, 3)
}).sort_values('importance', ascending=False)

# Plot a bar chart for visualisation
importances.head(25).plot.bar(figsize=(12, 4), x = 'feature', y='importance')
plt.show()

# Feature only the top features in X (train and Test)
top_features = importances.head(25)['feature'].tolist()
X = X[top_features]
X_TEST = X_TEST[top_features]

### 9. Train ML Models

In [ ]:
KNN_Classifier = KNeighborsClassifier(n_jobs=-1)
KNN_Classifier.fit(X, Y)

LGR_Classifier = LogisticRegression(n_jobs=-1, random_state=7, max_iter=1000, class_weight='balanced')
LGR_Classifier.fit(X, Y)

BNB_Classifier = BernoulliNB()
BNB_Classifier.fit(X, Y)

DTC_Classifier = tree.DecisionTreeClassifier(criterion='entropy', random_state=7, class_weight='balanced', max_depth=10, min_samples_leaf=20)
DTC_Classifier.fit(X, Y)

XGB_Classifier = XGBClassifier(random_state=7, n_estimators=200, max_depth=8, learning_rate=0.1, objective="multi:softmax", eval_metric="mlogloss")

### 10. Evaluate Model (on training data)

In [34]:
models = []
models.append(('Naive Bayes', BNB_Classifier))
models.append(('Decision Tree', DTC_Classifier))
models.append(('KNN', KNN_Classifier))
models.append(('Logistic Regression', LGR_Classifier))
models.append(("XGBoost", XGB_Classifier))

for name, model in models:
    cv_scores = cross_val_score(model, X, Y, cv=10)
    accuracy = metrics.accuracy_score(Y, model.predict(X))
    conf_matrix = metrics.confusion_matrix(Y, model.predict(X))
    report = metrics.classification_report(Y, model.predict(X))

    print(f"\n===== Train Evaluation =====")
    print("Cross Validation Mean Score:", cv_scores.mean())
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)


===== Train Evaluation =====
Cross Validation Mean Score: 0.5855146262214538
Accuracy: 0.5901087995295156
Confusion Matrix:
 [[28947  8961   659  2242]
 [13828 12458  5468  9055]
 [   11  1095 36616  3087]
 [ 2797  9574 10132 18306]]
Classification Report:
               precision    recall  f1-score   support

           0       0.64      0.71      0.67     40809
           1       0.39      0.31      0.34     40809
           2       0.69      0.90      0.78     40809
           3       0.56      0.45      0.50     40809

    accuracy                           0.59    163236
   macro avg       0.57      0.59      0.57    163236
weighted avg       0.57      0.59      0.57    163236


===== Train Evaluation =====
Cross Validation Mean Score: 0.6542193628553135
Accuracy: 0.6741098777230513
Confusion Matrix:
 [[29816  5777    56  5160]
 [11554 10476  2290 16489]
 [    1   277 38158  2373]
 [ 2116  2827  4277 31589]]
Classification Report:
               precision    recall  f1-score   s

### Valiate Model (on test data)

In [35]:
for name, model in models:
    accuracy = metrics.accuracy_score(Y_TEST, model.predict(X_TEST))
    conf_matrix = metrics.confusion_matrix(Y_TEST, model.predict(X_TEST))
    report = metrics.classification_report(Y_TEST, model.predict(X_TEST))

    print(f"\n===== {name} - Validation Results =====")
    print("Accuracy:", accuracy)
    print("Confusion Matrix:\n", conf_matrix)
    print("Classification Report:\n", report)


===== Naive Bayes - Validation Results =====
Accuracy: 0.46285
Confusion Matrix:
 [[ 880  506   23  262]
 [1397 1607  659 1731]
 [   0   28 2254  451]
 [ 714 2369 2603 4516]]
Classification Report:
               precision    recall  f1-score   support

           0       0.29      0.53      0.38      1671
           1       0.36      0.30      0.32      5394
           2       0.41      0.82      0.54      2733
           3       0.65      0.44      0.53     10202

    accuracy                           0.46     20000
   macro avg       0.43      0.52      0.44     20000
weighted avg       0.51      0.46      0.46     20000


===== Decision Tree - Validation Results =====
Accuracy: 0.5866
Confusion Matrix:
 [[ 758  364    4  545]
 [1123  965  325 2981]
 [   0    0 2301  432]
 [ 554  771 1169 7708]]
Classification Report:
               precision    recall  f1-score   support

           0       0.31      0.45      0.37      1671
           1       0.46      0.18      0.26      5394
 

### Train and Evaluate DL Model (ANN)

In [36]:
from keras.models import Sequential
from keras.layers import Dense
from keras.utils import to_categorical

n_classes = len(le.classes_)
Y_cat = to_categorical(Y, num_classes=n_classes)
Y_TEST_cat = to_categorical(Y_TEST, num_classes=n_classes)

# model = Sequential()
# model.add(Dense(64, activation='relu', input_shape=(X.shape[1],)))
# model.add(Dense(32, activation='relu'))
# model.add(Dense(n_classes, activation='softmax'))

## Make DL architecture deeper
model = Sequential([
    Dense(256, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(n_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X, Y_cat, epochs=20, batch_size=32, validation_split=0.1)

pred_probs = model.predict(X_TEST)
pred_classes = np.argmax(pred_probs, axis=1)

print("Accuracy:", metrics.accuracy_score(Y_TEST, pred_classes))
print(metrics.classification_report(Y_TEST, pred_classes))

Epoch 1/20
4591/4591 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.6372 - loss: 0.7761 - val_accuracy: 0.9632 - val_loss: 0.2276
Epoch 2/20
4591/4591 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.6581 - loss: 0.7321 - val_accuracy: 0.9639 - val_loss: 0.2199
Epoch 3/20
4591/4591 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.6642 - loss: 0.7214 - val_accuracy: 0.9670 - val_loss: 0.2105
Epoch 4/20
4591/4591 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.6667 - loss: 0.7156 - val_accuracy: 0.9661 - val_loss: 0.2120
Epoch 5/20
4591/4591 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.6691 - loss: 0.7113 - val_accuracy: 0.9669 - val_loss: 0.2072
Epoch 6/20
4591/4591 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.6711 - loss: 0.7076 - val_accuracy: 0.9715 - val_loss: 0.1940
Epoch 7/20
4591/4591 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.6724 - loss: 0.7047 - val_accuracy: 0.9716 - val_loss: 0.1871
Epoch 8/20
4591/4591 ━━━━━━━━━━━━━━━━━━━━ 9s 1ms/step - accuracy: 0.6739 - loss: 0.7023 - 